# БВК: поиск аналогов и классификация чертежей — обучение 3 моделей
**Настройки ноутбука (панель справа → Session options):** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = любой.
Датасет `bvk-drawings` добавить через **+ Add Input** (справа вверху) → Your Datasets.

**Запуск в фоне:** кнопка **Save Version → Save & Run All (Commit)**. Вкладку можно закрыть, через ~1–1.5 ч
результаты появятся во вкладке **Output** этой версии. Скачать `results_light.zip` (метрики и графики).

Модели: **DINOv2 ViT-B/14** (Meta, Apache-2.0) · **ruCLIP ViT-L/14-336** (Sber AI, Apache-2.0, отечественная) · **ResNet50** (baseline).
Для каждой: baseline без дообучения → дообучение ArcFace. Режим `full` — 8 классов; `open` — один класс скрыт (проверка «нового класса»).

In [ ]:
import torch, subprocess, sys, os
print(torch.__version__, '| GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), 'GPU не включён: Session options -> Accelerator -> GPU T4 x2'
p = torch.cuda.get_device_properties(0)
assert p.major >= 7, f'{p.name}: эта карта не подходит (P100 не поддерживается новым PyTorch). Выбери GPU T4 x2'

In [ ]:
!pip install -q -U timm pytorch-metric-learning 2>&1 | grep -v -i warning | tail -2
import timm, pytorch_metric_learning; print('timm', timm.__version__, '| pml', pytorch_metric_learning.__version__)

In [ ]:
%%writefile ruclip_vision.py
"""Визуальный энкодер ruCLIP (Sber AI / ai-forever, Apache-2.0) без пакета `ruclip`.

Зачем: `pip install ruclip` тянет huggingface-hub==0.2.1 и youtokentome — на Windows
и Python 3.10+ это обычно не ставится. Для чертежей нужен только визуальный энкодер,
текстовая часть не используется. Архитектура ниже повторяет ruclip/model.py (v0.0.2),
имена весов совпадают с чекпойнтом (префикс "visual.").

Использование:
    from ruclip_vision import load_ruclip_visual
    model = load_ruclip_visual("models/ruclip-vit-large-patch14-336")  # папка с config.json и pytorch_model.bin
    emb = model(x)   # x: [B,3,336,336], нормализация CLIP_MEAN/CLIP_STD
"""
import json
import os
from collections import OrderedDict

import torch
from torch import nn
from torch.utils.checkpoint import checkpoint

REPO_ID = "ai-forever/ruclip-vit-large-patch14-336"
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


class QuickGELU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(1.702 * x)


class ResidualAttentionBlock(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads)
        self.ln_1 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(OrderedDict([
            ("c_fc", nn.Linear(d, d * 4)), ("gelu", QuickGELU()), ("c_proj", nn.Linear(d * 4, d))]))
        self.ln_2 = nn.LayerNorm(d)

    def forward(self, x):
        h = self.ln_1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.mlp(self.ln_2(x))


class Transformer(nn.Module):
    def __init__(self, width, layers, heads):
        super().__init__()
        self.resblocks = nn.Sequential(*[ResidualAttentionBlock(width, heads) for _ in range(layers)])
        self.grad_checkpointing = False

    def forward(self, x):
        for blk in self.resblocks:
            if self.grad_checkpointing and self.training:
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)
        return x


class VisionTransformer(nn.Module):
    def __init__(self, input_resolution, patch_size, width, layers, heads, output_dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.output_dim = output_dim
        self.conv1 = nn.Conv2d(3, width, kernel_size=patch_size, stride=patch_size, bias=False)
        scale = width ** -0.5
        self.class_embedding = nn.Parameter(scale * torch.randn(width))
        self.positional_embedding = nn.Parameter(scale * torch.randn((input_resolution // patch_size) ** 2 + 1, width))
        self.ln_pre = nn.LayerNorm(width)
        self.transformer = Transformer(width, layers, heads)
        self.ln_post = nn.LayerNorm(width)
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))

    def set_grad_checkpointing(self, enable=True):
        self.transformer.grad_checkpointing = enable

    def forward(self, x):
        x = self.conv1(x)
        x = x.flatten(2).permute(0, 2, 1)
        cls = self.class_embedding.to(x.dtype).expand(x.shape[0], 1, -1)
        x = torch.cat([cls, x], dim=1) + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x).permute(1, 0, 2)          # NLD -> LND
        x = self.transformer(x).permute(1, 0, 2)    # LND -> NLD
        return self.ln_post(x[:, 0, :]) @ self.proj


def load_ruclip_visual(folder):
    cfg = json.load(open(os.path.join(folder, "config.json"), encoding="utf-8"))
    width = cfg["vision_width"]
    model = VisionTransformer(cfg["image_resolution"], cfg["vision_patch_size"], width,
                              cfg["vision_layers"], width // 64, cfg["embed_dim"])
    sd = torch.load(os.path.join(folder, "pytorch_model.bin"), map_location="cpu")
    vis = {k[len("visual."):]: v for k, v in sd.items() if k.startswith("visual.")}
    model.load_state_dict(vis, strict=True)   # упадёт с понятной ошибкой, если веса не те
    return model.float()


In [ ]:
%%writefile train.py
"""Обучение и оценка 3 моделей для поиска аналогов и классификации чертежей.

Схема (одинакова для всех моделей):
  изображение видов -> backbone -> эмбеддинг (L2-норм.) -> kNN по базе (галерее) ->
  класс голосованием top-k, аналоги = top-5 соседей, «новый класс» = сходство top-1 ниже порога.

Для каждой модели:
  1) baseline: предобученная модель без дообучения (kNN по её признакам);
  2) fine-tune: дообучение последних блоков + проекции 512 с ArcFace-лоссом.

Режимы (--run):
  full — 8 классов; основные метрики (Recall@1/5, accuracy, macro-F1, матрица ошибок) + модель для демо;
  open — один класс (--holdout) полностью скрыт при обучении и в галерее: проверка «нового класса» (AUROC).

Запуск:  python train.py --data <папка со split.csv> --model dinov2 --run full
Модели:  dinov2 | ruclip | resnet50 | siglip2
"""
import argparse, json, math, os, random, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
os.environ.setdefault("HF_HOME", str(HERE / "models" / "hf"))

IMNET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
CLIPN = ((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
HALF = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

# name: (вход, нормализация, батч T4/2070, учимых последних блоков (None = вся сеть), lr backbone, epochs)
CFG = {
    "dinov2":   dict(size=448, norm=IMNET, bs=16, last_k=6, lr=2e-5, epochs=15),
    "ruclip":   dict(size=336, norm=CLIPN, bs=16, last_k=4, lr=1e-5, epochs=15),
    "resnet50": dict(size=448, norm=IMNET, bs=32, last_k=None, lr=1e-4, epochs=20),
    "siglip2":  dict(size=384, norm=HALF, bs=16, last_k=4, lr=1e-5, epochs=15),
}


# ============================================================ модели
def build_backbone(name, pretrained=True):
    import timm
    if name == "dinov2":
        m = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=pretrained, num_classes=0, img_size=448)
        return m, m.blocks, m.num_features
    if name == "resnet50":
        m = timm.create_model("resnet50.a1_in1k", pretrained=pretrained, num_classes=0)
        return m, None, m.num_features
    if name == "siglip2":
        m = timm.create_model("vit_base_patch16_siglip_384.v2_webli", pretrained=pretrained, num_classes=0)
        return m, m.blocks, m.num_features
    if name == "ruclip":
        from ruclip_vision import load_ruclip_visual, VisionTransformer, REPO_ID
        if pretrained:
            d = HERE / "models" / "ruclip-vit-large-patch14-336"
            if not (d / "pytorch_model.bin").exists():
                from huggingface_hub import snapshot_download
                snapshot_download(REPO_ID, local_dir=str(d), allow_patterns=["config.json", "pytorch_model.bin"])
            m = load_ruclip_visual(str(d))
        else:  # для отладки без интернета
            m = VisionTransformer(336, 14, 1024, 24, 16, 768)
        return m, m.transformer.resblocks, m.output_dim
    raise ValueError(name)


class Net(nn.Module):
    """backbone -> (опционально) проекция 512 -> L2-нормированный эмбеддинг."""
    def __init__(self, name, pretrained=True, emb=512):
        super().__init__()
        self.backbone, self.blocks, feat = build_backbone(name, pretrained)
        self.proj = nn.Sequential(nn.Linear(feat, emb), nn.BatchNorm1d(emb))
        self.use_proj = False      # baseline: сырые признаки backbone; после fine-tune — проекция

    def forward(self, x):
        f = self.backbone(x).float()
        if self.use_proj:
            f = self.proj(f)
        return F.normalize(f, dim=-1)

    def set_trainable(self, last_k, ckpt=False):
        for p in self.backbone.parameters():
            p.requires_grad = last_k is None
        if last_k is not None:
            for b in list(self.blocks)[-last_k:]:
                for p in b.parameters():
                    p.requires_grad = True
            for n, p in self.backbone.named_parameters():   # финальные нормы/проекции после блоков
                if "blocks" not in n and any(s in n for s in ("ln_post", "proj", "norm", "head", "attn_pool")):
                    p.requires_grad = True
            if ckpt and hasattr(self.backbone, "set_grad_checkpointing"):   # экономия VRAM ценой ~30% скорости
                self.backbone.set_grad_checkpointing(True)


# ============================================================ данные
class Thicken:   # разная толщина линий: имитация другого CAD/сканера
    def __call__(self, im):
        r = random.random()
        if r < 0.25: return im.filter(ImageFilter.MinFilter(3))   # линии толще (фон белый)
        if r < 0.35: return im.filter(ImageFilter.MaxFilter(3))   # тоньше
        return im


class DrawingSet(Dataset):
    def __init__(self, root, df, size, norm, train):
        import torchvision.transforms as T
        self.labels = df.label.values
        self.imgs = [Image.open(root / p).convert("L").copy() for p in df.file_view]   # ~650 МБ в RAM, быстро
        ops = []
        if train:
            ops += [Thicken(),
                    T.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.9, 1.08), fill=255),
                    T.RandomHorizontalFlip(),
                    T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 0.8))], p=0.3)]
        ops += [T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC, antialias=True),
                T.Grayscale(3), T.ToTensor(), T.Normalize(*norm)]
        self.tf = T.Compose(ops)

    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tf(self.imgs[i]), self.labels[i]


@torch.no_grad()
def embed(model, loader, dev):
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
            out.append(model(x.to(dev, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()


# ============================================================ метрики
def knn(q, g, k=10):
    s = q @ g.T
    idx = np.argsort(-s, axis=1)[:, :k]
    return idx, np.take_along_axis(s, idx, 1)


def evaluate(q, ql, g, gl, n_cls, k_vote=5):
    from sklearn.metrics import f1_score, confusion_matrix
    idx, sim = knn(q, g, 10)
    nl = gl[idx]
    r1 = float((nl[:, 0] == ql).mean())
    r5 = float((nl[:, :5] == ql[:, None]).any(1).mean())
    votes = np.zeros((len(q), n_cls))
    for j in range(k_vote):
        np.add.at(votes, (np.arange(len(q)), nl[:, j]), np.exp(sim[:, j] / 0.1))   # взвешенное голосование
    pred = votes.argmax(1)
    labels = sorted(set(ql.tolist()))
    return dict(recall_at_1=r1, recall_at_5=r5,
                accuracy=float((pred == ql).mean()),
                macro_f1=float(f1_score(ql, pred, labels=labels, average="macro", zero_division=0)),
                cm=confusion_matrix(ql, pred, labels=list(range(n_cls))).tolist(),
                pred=pred, top1_sim=sim[:, 0], nn_idx=idx, nn_sim=sim)


def open_set(score_known_val, score_unk_val, score_known_test, score_unk_test):
    """score = сходство с ближайшим аналогом. Порог выбираем на val (95% известных проходят), проверяем на test."""
    from sklearn.metrics import roc_auc_score
    y = np.r_[np.ones(len(score_known_test)), np.zeros(len(score_unk_test))]
    auroc = roc_auc_score(y, np.r_[score_known_test, score_unk_test])
    thr = float(np.quantile(score_known_val, 0.05))
    return dict(auroc_new_class=float(auroc), threshold=thr,
                known_kept=float((score_known_test >= thr).mean()),
                new_detected=float((score_unk_test < thr).mean()))


# ============================================================ графики
def plot_cm(cm, names, title, path):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cm = np.array(cm); n = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.imshow(n, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(names)):
        for j in range(len(names)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9, color="white" if n[i, j] > .5 else "black")
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(names)), names, fontsize=8)
    ax.set_xlabel("предсказано"); ax.set_ylabel("истина"); ax.set_title(title, fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)


def plot_analogs(root, qdf, gdf, res, names, path, n=8, seed=0):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    rng = np.random.default_rng(seed); rows = rng.choice(len(qdf), size=min(n, len(qdf)), replace=False)
    fig, axs = plt.subplots(len(rows), 6, figsize=(12, 2.1 * len(rows)))
    for r, qi in enumerate(rows):
        ax = axs[r, 0]; ax.imshow(Image.open(root / qdf.file_view.iloc[qi]), cmap="gray")
        ax.set_title(f"ЗАПРОС\n{names[qdf.label.iloc[qi]][:18]}", fontsize=7, color="navy")
        for c in range(5):
            gi = res["nn_idx"][qi, c]; ok = gdf.label.iloc[gi] == qdf.label.iloc[qi]
            ax = axs[r, c + 1]; ax.imshow(Image.open(root / gdf.file_view.iloc[gi]), cmap="gray")
            ax.set_title(f"#{c+1} sim={res['nn_sim'][qi, c]:.2f}\n{names[gdf.label.iloc[gi]][:18]}",
                         fontsize=7, color="green" if ok else "red")
        for ax in axs[r]: ax.axis("off")
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ============================================================ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="папка, где лежат split.csv и png_views/")
    ap.add_argument("--model", required=True, choices=list(CFG))
    ap.add_argument("--run", default="full", choices=["full", "open"])
    ap.add_argument("--holdout", default="06_petli_kronshteyny")
    ap.add_argument("--out", default="results")
    ap.add_argument("--epochs", type=int); ap.add_argument("--bs", type=int)
    ap.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 2))
    ap.add_argument("--ckpt", action="store_true", help="gradient checkpointing (нужно на 8 ГБ для ruclip)")
    ap.add_argument("--no_finetune", action="store_true", help="только baseline")
    ap.add_argument("--smoke", action="store_true", help="отладка на CPU: без весов, мало данных")
    a = ap.parse_args()

    cfg = dict(CFG[a.model]); cfg["epochs"] = a.epochs or cfg["epochs"]; cfg["bs"] = a.bs or cfg["bs"]
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    root = Path(a.data)
    out = Path(a.out) / f"{a.model}_{a.run}"; out.mkdir(parents=True, exist_ok=True)
    log = open(out / "log.txt", "a", encoding="utf-8")
    def say(*s):
        msg = " ".join(str(x) for x in s); print(msg, flush=True); log.write(msg + "\n"); log.flush()

    df = pd.read_csv(root / "split.csv")
    df = df[df.split.isin(["train", "val", "test"])]
    df = df[df.file_view.map(lambda p: (root / p).exists())].reset_index(drop=True)
    names = sorted(df.our_class.unique()); n_cls = len(names)
    df["label"] = df.our_class.map({c: i for i, c in enumerate(names)})
    if a.smoke:
        df = df.groupby(["our_class", "split"]).head(4).reset_index(drop=True); cfg["epochs"] = 1; cfg["bs"] = 4
    hold = names.index(a.holdout) if a.run == "open" else -1
    known = df[df.label != hold]
    tr, va, te = (known[known.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    unk_va = df[(df.label == hold) & (df.split == "val")].reset_index(drop=True)
    unk_te = df[(df.label == hold) & (df.split == "test")].reset_index(drop=True)
    say(f"=== {a.model} / {a.run} | устройство {dev} | train {len(tr)} val {len(va)} test {len(te)}"
        + (f" | скрытый класс {a.holdout}: val {len(unk_va)} test {len(unk_te)}" if hold >= 0 else ""))

    S, N = cfg["size"], cfg["norm"]
    mk = lambda d, t: DrawingSet(root, d, S, N, t)
    dl = lambda ds, sh=False, sampler=None: DataLoader(ds, batch_size=cfg["bs"] * (1 if sampler else 2), shuffle=sh,
                                                       sampler=sampler, num_workers=a.workers, pin_memory=dev == "cuda",
                                                       drop_last=sampler is not None, persistent_workers=a.workers > 0)
    ev = {k: dl(mk(d, False)) for k, d in dict(tr=tr, va=va, te=te, uva=unk_va, ute=unk_te).items() if len(d)}

    model = Net(a.model, pretrained=not a.smoke).to(dev)

    def full_eval(tag):
        E = {k: embed(model, l, dev) for k, l in ev.items()}
        gal = np.r_[E["tr"], E["va"]]; gl = np.r_[tr.label.values, va.label.values]
        gdf = pd.concat([tr, va], ignore_index=True)
        r = evaluate(E["te"], te.label.values, gal, gl, n_cls)
        m = {k: v for k, v in r.items() if k not in ("pred", "top1_sim", "nn_idx", "nn_sim")}
        if hold >= 0:   # порог по val (галерея = train), проверка на test (галерея = train+val)
            rv = evaluate(E["va"], va.label.values, E["tr"], tr.label.values, n_cls)
            su_va = knn(E["uva"], E["tr"], 1)[1][:, 0]; su_te = knn(E["ute"], gal, 1)[1][:, 0]
            m.update(open_set(rv["top1_sim"], su_va, r["top1_sim"], su_te))
        cls_names = [n for i, n in enumerate(names) if i != hold] if hold >= 0 else names
        cm = np.array(m["cm"]); keep = [i for i in range(n_cls) if i != hold]
        plot_cm(cm[np.ix_(keep, keep)] if hold >= 0 else cm, cls_names,
                f"{a.model} {tag}: acc={m['accuracy']:.3f}  R@1={m['recall_at_1']:.3f}", out / f"cm_{tag}.png")
        plot_analogs(root, te, gdf, r, names, out / f"analogs_{tag}.png")
        json.dump(m, open(out / f"metrics_{tag}.json", "w"), indent=1)
        say(f"[{tag}] " + "  ".join(f"{k}={v:.4f}" for k, v in m.items() if isinstance(v, float)))
        return m, E, gdf

    t0 = time.time()
    base, _, _ = full_eval("baseline")
    results = {"baseline": base}

    if not a.no_finetune:
        model.set_trainable(cfg["last_k"], ckpt=a.ckpt); model.use_proj = True
        from pytorch_metric_learning.losses import ArcFaceLoss
        n_known = n_cls  # метки сохраняем исходные; скрытый класс просто не встречается
        crit = ArcFaceLoss(num_classes=n_known, embedding_size=512, margin=28.6, scale=30).to(dev)
        bb = [p for p in model.backbone.parameters() if p.requires_grad]
        opt = torch.optim.AdamW([{"params": bb, "lr": cfg["lr"]},
                                 {"params": model.proj.parameters(), "lr": 1e-3},
                                 {"params": crit.parameters(), "lr": 1e-2}], weight_decay=0.05)
        cnt = tr.label.value_counts(); w = 1.0 / cnt[tr.label].values        # баланс классов
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(tr), replacement=True)
        trl = dl(mk(tr, True), sampler=sampler)
        steps = cfg["epochs"] * len(trl); warm = len(trl)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))
        scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda")
        say(f"fine-tune: учим {sum(p.numel() for p in bb) / 1e6:.1f}M параметров backbone, "
            f"батч {cfg['bs']}, эпох {cfg['epochs']}, шагов/эпоху {len(trl)}")
        best, best_ep, patience = -1, 0, 5
        for ep in range(1, cfg["epochs"] + 1):
            model.train(); te0 = time.time(); tot = 0
            for x, y in trl:
                x, y = x.to(dev, non_blocking=True), y.to(dev)
                with torch.autocast("cuda", dtype=torch.float16, enabled=dev == "cuda"):
                    f = model.proj(model.backbone(x).float())
                loss = crit(f.float(), y)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 5.0)
                scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
            # валидация: val-запросы против train-галереи
            Etr, Eva = embed(model, ev["tr"], dev), embed(model, ev["va"], dev)
            v = evaluate(Eva, va.label.values, Etr, tr.label.values, n_cls)
            score = v["recall_at_1"] + v["macro_f1"]
            say(f"эпоха {ep:2d}  loss {tot / len(trl):.3f}  val R@1 {v['recall_at_1']:.4f}  "
                f"val F1 {v['macro_f1']:.4f}  {time.time() - te0:.0f} c")
            if score > best:
                best, best_ep = score, ep
                torch.save({"model": a.model, "state": {k: v.half() if v.is_floating_point() else v
                                                        for k, v in model.state_dict().items()},
                            "classes": names, "cfg": cfg}, out / "best.pt")
            elif ep - best_ep >= patience:
                say(f"ранняя остановка: {patience} эпох без улучшения"); break
        ck = torch.load(out / "best.pt", map_location="cpu")
        model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in ck["state"].items()})
        say(f"лучшая эпоха {best_ep}")
        ft, E, gdf = full_eval("finetuned")
        results["finetuned"] = ft; results["best_epoch"] = best_ep
        if a.run == "full":    # база для демо: эмбеддинги всех известных чертежей + путь к файлам
            np.savez_compressed(out / "gallery.npz", emb=np.r_[E["tr"], E["va"], E["te"]].astype(np.float16),
                                files=np.r_[gdf.file_view.values, te.file_view.values].astype(str),
                                labels=np.r_[gdf.label.values, te.label.values], classes=np.array(names, dtype=str))

    results.update(model=a.model, run=a.run, holdout=a.holdout if hold >= 0 else None,
                   minutes=round((time.time() - t0) / 60, 1), cfg={k: v for k, v in cfg.items() if k != "norm"})
    json.dump(results, open(out / "summary.json", "w"), indent=1, ensure_ascii=False)
    say(f"готово за {results['minutes']} мин -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
import glob
hits = glob.glob('/kaggle/input/**/split.csv', recursive=True)
assert hits, 'Датасет не подключён: + Add Input -> Your Datasets -> bvk-drawings'
DATA = os.path.dirname(hits[0]); print('DATA =', DATA)
import pandas as pd; s = pd.read_csv(hits[0]); print(pd.crosstab(s.our_class, s.split))

## Обучение
На 2×T4 две очереди идут параллельно (по одной на видеокарту). `EPOCHS=None` — значения по умолчанию из train.py
(15 для ViT, 20 для ResNet50, с ранней остановкой). Для быстрой пробы поставь `EPOCHS = 2`.

In [ ]:
MODELS_Q = [  # очередь на GPU0, очередь на GPU1
    [('dinov2', 'full'), ('dinov2', 'open'), ('resnet50', 'full'), ('resnet50', 'open')],
    [('ruclip', 'full'), ('ruclip', 'open')],
]
HOLDOUT = '06_petli_kronshteyny'   # класс, скрываемый в режиме open
EPOCHS = None
import time, threading
ngpu = torch.cuda.device_count()
if ngpu < 2: MODELS_Q = [MODELS_Q[0] + MODELS_Q[1]]
os.makedirs('results', exist_ok=True)

def run_queue(gi, q):
    for m, r in q:
        cmd = [sys.executable, 'train.py', '--data', DATA, '--model', m, '--run', r, '--holdout', HOLDOUT,
               '--out', 'results', '--workers', '2'] + (['--epochs', str(EPOCHS)] if EPOCHS else [])
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi))
        with open(f'results/console_{m}_{r}.txt', 'w') as f:
            rc = subprocess.call(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        print(f'[GPU{gi}] {m}/{r}: ' + ('OK' if rc == 0 else f'ОШИБКА (код {rc}) — см. results/console_{m}_{r}.txt'), flush=True)

t0 = time.time()
th = [threading.Thread(target=run_queue, args=(i, q)) for i, q in enumerate(MODELS_Q)]
[t.start() for t in th]
while any(t.is_alive() for t in th):   # раз в 2 минуты — последние строки логов
    time.sleep(120)
    for f in sorted(glob.glob('results/*/log.txt')):
        last = open(f, encoding='utf-8').read().strip().splitlines()[-1:]
        print(f'{time.time()-t0:5.0f}s {f.split("/")[1]:16s} {last[0] if last else ""}'[:160], flush=True)
[t.join() for t in th]
print(f'Всё обучение: {(time.time()-t0)/60:.1f} мин')

## Сводка результатов

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
rows = []
for f in sorted(glob.glob('results/*/summary.json')):
    s = json.load(open(f))
    for stage in ('baseline', 'finetuned'):
        if stage not in s: continue
        m = s[stage]
        rows.append({'модель': s['model'], 'режим': s['run'], 'этап': stage,
                     'Recall@1': m['recall_at_1'], 'Recall@5': m['recall_at_5'], 'Accuracy': m['accuracy'],
                     'Macro-F1': m['macro_f1'], 'AUROC нов.класс': m.get('auroc_new_class'),
                     'нов.класс найден': m.get('new_detected'), 'известн. сохранены': m.get('known_kept'),
                     'мин': s['minutes']})
T = pd.DataFrame(rows).round(3)
T.to_csv('results/summary_all.csv', index=False, encoding='utf-8-sig')
display(T)
full = T[T['режим'] == 'full'].pivot(index='модель', columns='этап', values=['Recall@1', 'Macro-F1'])
ax = full.plot.bar(figsize=(9, 4), rot=0, ylim=(0, 1), title='8 классов: до и после дообучения')
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig('results/compare_full.png', dpi=130); plt.show()
op = T[(T['режим'] == 'open')].pivot(index='модель', columns='этап', values='AUROC нов.класс')
if len(op):
    ax = op.plot.bar(figsize=(7, 3.5), rot=0, ylim=(0.4, 1), title=f'Распознавание нового класса (скрыт {HOLDOUT}), AUROC')
    plt.tight_layout(); plt.savefig('results/compare_open.png', dpi=130); plt.show()

In [ ]:
from IPython.display import Image as Im, display
for f in sorted(glob.glob('results/*_full/cm_finetuned.png')) + sorted(glob.glob('results/*_full/analogs_finetuned.png'))[:1]:
    print(f); display(Im(f, width=700))

In [ ]:
# лёгкий архив: метрики, графики, логи (без весов); веса best.pt и gallery.npz лежат в Output отдельно
!cd results && zip -q -r ../results_light.zip . -x '*.pt' '*.npz' && cd .. && ls -la results_light.zip
!du -sh results/*/best.pt 2>/dev/null